## Project

Have different LLMs generate answers to a question and let all them be judged and ranked by an LLM only

In [4]:
import os
from openai import OpenAI
import json
from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import display , Markdown

In [5]:
load_dotenv(override=True)

openai = OpenAI()

In [6]:
#Load all API keys

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

In [ ]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{'role':'user','content':request}]
messages

In [8]:
response = openai.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages
    )

question=response.choices[0].message.content

print(question)

How would you approach the ethical implications of employing artificial intelligence in decision-making processes that significantly impact human lives, considering the potential biases embedded in the data and the transparency of the algorithms used?


In [9]:
competitors = []
answers=[]
messages = [{'role':'user','content':question}]

In [ ]:
#OpenIA

model_name='gpt-4o-mini'

response = openai.chat.completions.create(
    model=model_name, messages=messages)
answer=response.choices[0].message.content

display(Markdown(answer))

competitors.append(model_name)
answers.append(answer)



In [ ]:
#Anthropic

model_name="claude-3-7-sonnet-latest"
claude=Anthropic()

response=claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer=response.choices[0].text

display(Markdown(answer))

competitors.append(model_name)
answers.append(answer)

In [ ]:
#Gemini

gemini=OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

model_name= "gemini-2.0-flash"

response=gemini.chat.completions.create(model=model_name, messages=messages)
answer=response.choices[0].message.content

display(Markdown(answer))

competitors.append(model_name)
answers.append(answer)

In [ ]:
#Deepseek

deepseek=OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")

model_name= "deepseek-chat"

response=deepseek.chat.completions.create(model=model_name, messages=messages)
answer=response.choices[0].message.content

display(Markdown(answer))

competitors.append(model_name)
answers.append(answer)

In [ ]:
#Groq

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "llama-3.3-70b-versatile"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


In [ ]:
!ollama pull llama3.2

In [ ]:
#Ollama

ollama=OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

model_name="llama3.2"

response=ollama.chat.completions.create(model=model_name, messages=messages)
answer=response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [20]:
print(competitors)
print(answers)

['gpt-4o-mini', 'gemini-2.0-flash', 'llama3.2']
['Addressing the ethical implications of employing artificial intelligence (AI) in decision-making processes that significantly impact human lives involves a multifaceted approach. Here are several key areas to consider:\n\n### 1. **Understanding and Mitigating Bias**\n   - **Data Audit**: Conduct thorough audits of the data used to train AI systems to identify potential biases. This includes examining the sources of data, the population represented, and any historical biases that may be embedded within it.\n   - **Diverse Datasets**: Use diverse and representative datasets to mitigate biases. This could involve collecting more data from underrepresented groups or employing synthetic data where necessary.\n   - **Bias Detection Tools**: Implement tools and frameworks designed to detect and measure bias in AI outputs. Regularly test algorithms against these metrics.\n\n### 2. **Transparency in Algorithms**\n   - **Explainability**: Develop

In [ ]:
for competitor, answer in zip(competitors, answers):
    print(f"Competitors: {competitor}\n\n{answer}")


In [33]:
total=""

for index, answer in enumerate(answers):
    total += f"Response from competitor {index+1}\n\n"
    total += answer +"\n\n"

In [ ]:
print(total)

In [34]:
# Now for the judgement part

judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

{total}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [35]:
judge_message=[{'role':'user','content':judge}]

In [36]:
openai=OpenAI()

response=openai.chat.completions.create(
    model="o3-mini", messages=judge_message
)
results = response.choices[0].message.content
print(results)

{"results": ["2", "1", "3"]}


In [ ]:
results_dict=json.loads(results)
ranks=results_dict['results']
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank: {index+1}:{competitor}")